# AlgoArena: Sharpe-first model, version 3

Run the cells in order. This notebook downloads the official competition data, trains and audits the model, and produces `predictions.csv`. Actual hidden-OOS Sharpe is only known after a manual leaderboard submission.

In [ ]:
from pathlib import Path
import subprocess, sys
repo = Path('/content/HRT-Game')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/ABCPeter123/HRT-Game.git', str(repo)], check=True)
else:
    print('Using existing checkout:', repo)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', str(repo / 'requirements.txt')], check=True)

In [ ]:
data_dir = repo / 'data'
data_dir.mkdir(exist_ok=True)
if not ((data_dir / 'sessions.py').exists() and list(data_dir.glob('ins_*.parquet')) and list(data_dir.glob('oos_*.parquet'))):
    url = 'https://drive.google.com/drive/folders/1mYOtE9R-XtdcJ4cPVpQYAxiPLD1Dm4vX?usp=sharing'
    subprocess.run([sys.executable, '-m', 'gdown', '--folder', url, '--output', str(data_dir)], check=True)
print('Competition data:', data_dir)
subprocess.run([sys.executable, str(repo / 'algoarena_sharpe_v3.py'), '--data-dir', str(data_dir), '--check-data'], check=True)

## Validate, audit, and generate predictions

Three index-ordered validation folds choose a Sharpe-first configuration. The sealed 20% audit is held aside until selection. Reports are saved in a unique experiment folder.

In [ ]:
from datetime import datetime, timezone
run_name = datetime.now(timezone.utc).strftime('colab_%Y%m%dT%H%M%SZ')
report_dir = repo / 'runs' / run_name
subprocess.run([sys.executable, str(repo / 'algoarena_sharpe_v3.py'),
                '--data-dir', str(data_dir), '--output-dir', str(report_dir)], check=True)
print('Reports:', report_dir)
print('Submission:', repo / 'predictions.csv')

In [ ]:
import json, pandas as pd
from IPython.display import display, Image
report = json.loads((report_dir / 'report.json').read_text())
print('Validation Sharpe:', round(report['selection']['oof_sharpe'], 3))
print('Sealed audit Sharpe:', round(report['audit']['sharpe'], 3))
print('Audit bootstrap CI:', report['audit']['sharpe_bootstrap_95pct_ci'])
print('Baseline Sharpes:', {k: round(v['sharpe'], 3) for k,v in report['baselines'].items()})
display(pd.read_csv(report_dir / 'candidate_scores.csv').head(10))
display(pd.read_csv(report_dir / 'audit_breakdowns.csv'))
display(Image(filename=str(report_dir / 'audit_equity_curve.png')))

In [ ]:
import shutil
archive = shutil.make_archive(str(report_dir), 'zip', root_dir=report_dir)
print('Submission:', repo / 'predictions.csv')
print('Debug bundle:', archive)
try:
    from google.colab import files
    files.download(str(repo / 'predictions.csv'))
    files.download(archive)
except ImportError:
    pass